# Dubai Real Estate Lead Analytics
## Notebook 01 — Raw Data Exploration

**Company:** A.S. Properties (Dubai-based real estate developer)
**Dataset:** HubSpot CRM export — Facebook lead campaigns, March 9–13 2026
**Analyst purpose:** Profile the raw data before cleaning to document what we have, what's broken, and what needs to be fixed.

---

This notebook examines the **raw, unmodified HubSpot export** from a 5-day Facebook lead generation campaign window. Before touching the data, we need to understand its structure, assess quality, and design a cleaning strategy. Rushing into analysis on uncleaned CRM data produces misleading metrics — a 169-lead dataset with 6 completely empty columns and mislabelled phone numbers is a liability, not an asset.

**Questions this notebook answers:**
1. What is the exact structure of the raw dataset — columns, types, completeness?
2. Which columns have data quality problems, and how severe are they?
3. What cleaning operations are needed before any business analysis can begin?


In [1]:
import os
import sys

os.chdir('..')
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from src import config
from src.visualization import save_fig

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Setup complete. Project root:", os.getcwd())


Setup complete. Project root: C:\Projects\Real Estate Analytics


## 1. Loading the Raw Dataset

We load only the **first sheet** of the Excel file. HubSpot exports include a second sheet ("Sheet1") with a pivot table summary — we ignore it to avoid ingesting aggregated data as if it were raw records.


In [2]:
df = pd.read_excel(config.RAW_FILE, sheet_name=0, engine='openpyxl')

print(f"Shape: {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"File : {config.RAW_FILE.name}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")


Shape: 169 rows  x  22 columns
File : week 2 campaign performance.xlsx

Column names:
   1. Record ID
   2. First Name
   3. Last Name
   4. Phone Number
   5. Contact owner
   6. Lead Status
   7. Associated Note
   8. Create Date
   9. Original Source Drill-Down 2
  10. Email
  11. Recent Conversion
  12. Last Activity Date
  13. Marketing contact status
  14. Original Source
  15. Original Source Drill-Down 1
  16. Source 3
  17. Are you an investor or a broker
  18. Unit Type
  19. Unit Value
  20. Original Create Date
  21. Recent Deal Close Date
  22. Associated Note IDs


**What we have:** 169 leads captured over a 5-day window (March 9–13, 2026) across 9 Facebook campaigns. At 22 columns, the export looks comprehensive — but as we'll see shortly, 9 of those columns carry no usable information and will be dropped.

The column names immediately reveal two HubSpot conventions to watch for:
- **"Original Source Drill-Down 2"** is the actual Facebook campaign/ad name — the field name bears no resemblance to its content.
- **"Associated Note IDs"** and **"Associated Note"** are two distinct columns; only the latter contains the human-readable follow-up notes with D1/D2/D3 tags.


## 2. Column Overview: Data Types & Completeness

The first diagnostic is a single table covering every column: its dtype, how many values are present, and how many are missing. This instantly reveals structural problems.


In [3]:
summary = pd.DataFrame({
    'dtype'      : df.dtypes.astype(str),
    'non_null'   : df.notna().sum(),
    'null_count' : df.isna().sum(),
    'pct_null'   : (df.isna().mean() * 100).round(1),
    'unique'     : df.nunique(),
}).rename_axis('column').reset_index()

# Highlight fully-empty columns
def style_nulls(row):
    if row['pct_null'] == 100:
        return ['background-color: #ffcccc'] * len(row)
    elif row['pct_null'] >= 20:
        return ['background-color: #fff3cd'] * len(row)
    return [''] * len(row)

summary.style.apply(style_nulls, axis=1).format({'pct_null': '{:.1f}%'})


,column,dtype,non_null,null_count,pct_null,unique
0,Record ID,int64,169,0,0.0%,169
1,First Name,object,168,1,0.6%,159
2,Last Name,object,107,62,36.7%,107
3,Phone Number,float64,168,1,0.6%,167
4,Contact owner,object,165,4,2.4%,3
5,Lead Status,object,138,31,18.3%,9
6,Associated Note,object,135,34,20.1%,94
7,Create Date,datetime64[ns],169,0,0.0%,166
8,Original Source Drill-Down 2,object,169,0,0.0%,9
9,Email,object,168,1,0.6%,168


**Findings at a glance** (red rows = 100% empty, yellow = 20%+ missing):

| Problem | Columns | Impact |
|---------|---------|--------|
| **100% empty** | Source 3, investor/broker flag, Unit Type, Unit Value, Original Create Date, Recent Deal Close Date | 6 columns with zero information — drop immediately |
| **Zero variance** | Marketing contact status, Original Source, Original Source Drill-Down 1 | All 169 rows share the same value — no analytical use |
| **Phone as float64** | Phone Number | HubSpot stored `971501234567.0` — scientific notation risk and must be cast to string |
| **20% missing activity date** | Last Activity Date | 34 leads were never contacted after import — a significant operational gap |
| **18% missing lead status** | Lead Status | 31 leads have no status — ambiguous without context; we'll treat them as "Uncontacted" |

After dropping the 9 useless columns, we're left with **13 genuinely informative columns** to clean and analyse.


## 3. Sample Data

Seeing actual rows catches formatting problems that aggregate summaries miss — unusual encodings, mixed-type values, or truncated text fields.


In [4]:
print("First 5 rows:")
df.head()


First 5 rows:


,Record ID,First Name,Last Name,Phone Number,Contact owner,Lead Status,Associated Note,Create Date,Original Source Drill-Down 2,Email,Recent Conversion,Last Activity Date,Marketing contact status,Original Source,Original Source Drill-Down 1,Source 3,Are you an investor or a broker,Unit Type,Unit Value,Original Create Date,Recent Deal Close Date,Associated Note IDs
0,208956061013,Jet,Guy Jetlife,393511841860.00,NaN,NaN,NaN,2026-03-13 09:22:00,new project teaser - europe (en) lead gen a.s.,ganjabella12@gmail.com,Facebook Lead Ads: New Project - New lead gen form - A.S...,NaT,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,208962569084,Bhawani Singh,NaN,971524141674.00,NaN,NaN,NaN,2026-03-13 08:57:00,new - lead gen uae (en) - a.s. - copy,bhawanisingh06061985@gmail.com,Facebook Lead Ads: New lead gen form - A.S. (V6) - AED,NaT,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,208953719219,Khalid,Al Ali,971508877511.00,NaN,NaN,NaN,2026-03-13 08:43:00,new project teaser - uae (en) lead gen a.s.,boflan123456789@gmail.com,Facebook Lead Ads: New Project - New lead gen form - A.S...,NaT,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,208942961783,Sergio,A Rodrigues,33617529803.00,Ruqaia Hajalsiddig,NaN,NaN,2026-03-13 07:17:00,new project - optimized europe lookalike (en) lead gen a.s.,sergioantoniomoura@hotmail.com,Facebook Lead Ads: New Project - New lead gen form - A.S...,NaT,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,208940784707,Виктор,Матвеев,380668898568.00,Nouhed Hazem,No Answer,D1: wa sent,2026-03-13 07:10:00,new project - optimized europe lookalike (en) lead gen a.s.,viktormatveev568@gmail.com,Facebook Lead Ads: New Project - New lead gen form - A.S...,2026-03-13 08:30:00,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,106298132066


Two things stand out immediately from the first rows:
1. **Phone Number** shows as `9.715e+11` in the default view — exactly the scientific notation risk that makes the float→int→string conversion non-negotiable.
2. **Original Source Drill-Down 2** contains the full Facebook campaign name (e.g. `"new project teaser - europe (en) lead gen a.s."`) — this encodes region, campaign type, and language in a single string that we'll need to parse.


In [5]:
print("Last 5 rows:")
df.tail()


Last 5 rows:


,Record ID,First Name,Last Name,Phone Number,Contact owner,Lead Status,Associated Note,Create Date,Original Source Drill-Down 2,Email,Recent Conversion,Last Activity Date,Marketing contact status,Original Source,Original Source Drill-Down 1,Source 3,Are you an investor or a broker,Unit Type,Unit Value,Original Create Date,Recent Deal Close Date,Associated Note IDs
164,207953664800,Максат,Эргешов,996701303321.00,amr shaaban,No Answer,D1: line issues \ WA Sent,2026-03-09 04:40:00,new project - optimized europe lookalike (en) lead gen a.s.,maksatergesova@gmail.com,Facebook Lead Ads: New Project - New lead gen form - A.S...,2026-03-09 08:59:00,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,106000515477
165,207947172870,Svetlana,NaN,380930373964.00,Ruqaia Hajalsiddig,Unqualified,Never inquired;D1: Language barrier (Ukrainian) / Whatsa...,2026-03-09 04:15:00,lead gen eur (en) lookalike - a.s.,svtlnnedaskovskaa@gmail.com,Facebook Lead Ads: New lead gen form - A.S. (V4.1) - USD,2026-03-10 10:39:00,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,106084586892;106000737448
166,207935716148,Md,Rakib,971524649592.00,Nouhed Hazem,Junk Lead,registered by mistake,2026-03-09 02:47:00,new project teaser - uae (en) lead gen a.s.,rakibmd36906@gmail.com,Facebook Lead Ads: New Project - New lead gen form - A.S...,2026-03-09 10:16:00,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,106000530219
167,207929015591,salsh,NaN,971545177102.00,amr shaaban,Unqualified,registered by mistake,2026-03-09 02:10:00,new - lead gen uae (en) - a.s. - copy,samasteels@gmail.com,Facebook Lead Ads: New lead gen form - A.S. (V6) - AED,2026-03-09 10:03:00,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,106009207477
168,207922661970,Johnnie,Sheehan,353866086447.00,Ruqaia Hajalsiddig,No Answer,D3: No answer;D2: No answer / Whatsapp+ Email sent;D1: N...,2026-03-09 00:20:00,new project teaser - uk & ireland (en) lead gen a.s.,jonsheen9@gmail.com,Facebook Lead Ads: New Project - New lead gen form - A.S...,2026-03-11 11:05:00,Non-marketing contact,Paid Social,Facebook,NaN,NaN,NaN,NaN,NaN,NaN,106189145262;106074753859;106020642061


The last rows confirm the dataset ends on March 13, 2026, and that missing Lead Status values (NaN) are scattered throughout — not concentrated at the tail — suggesting they reflect unprocessed leads rather than a loading truncation issue.


## 4. Categorical Column Analysis

We examine every text-based column with value counts to understand distributions before any cleaning.


### 4.1 Lead Status

This is the most critical column for sales analysis. Its distribution reveals the team's overall pipeline health at a glance.


In [6]:
# Value counts including NaN
status_counts = df['Lead Status'].value_counts(dropna=False)
status_pct    = (status_counts / len(df) * 100).round(1)

print("Lead Status distribution (including NaN):")
for status, count in status_counts.items():
    label = str(status) if not pd.isna(status) else "(blank)"
    print(f"  {label:<30} {count:>4}  ({status_pct[status]:.1f}%)")

# Chart
plot_df = pd.DataFrame({
    'status': [str(s) if not pd.isna(s) else '(blank — Uncontacted)' for s in status_counts.index],
    'count' : status_counts.values
}).sort_values('count')

fig = px.bar(
    plot_df, x='count', y='status', orientation='h',
    text='count',
    color_discrete_sequence=[config.COLORS['primary']],
    title='Raw Lead Status Distribution (n=169)',
    labels={'count': 'Number of Leads', 'status': 'Lead Status'},
)
fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=180, r=60, t=60, b=40),
    height=420,
    xaxis_title='Number of Leads',
    yaxis_title='',
)
save_fig(fig, '01_raw_lead_status')
fig.show()


Lead Status distribution (including NaN):
  No Answer                        96  (56.8%)
  (blank)                          31  (18.3%)
  Contacted                        17  (10.1%)
  Unqualified                      14  (8.3%)
  Not Interested                    3  (1.8%)
  Future Opportunity                3  (1.8%)
  Junk Lead                         2  (1.2%)
  Newsletter Subscription           1  (0.6%)
  Qualified                         1  (0.6%)
  Hot Lead                          1  (0.6%)


**Business insight:** "No Answer" accounts for 57% of all leads — the single largest outcome by a wide margin. Combined with the 31 blank statuses (18%) which represent leads that were never even given a status, **75% of leads show no productive engagement**. The real question is whether "No Answer" reflects a lead quality problem (people submitting Facebook forms casually) or a response-time problem (calling too late for the lead to remember registering). Subsequent notebooks will test both hypotheses.

The 4 genuinely positive outcomes — Contacted (9), Future Opportunity (6), Hot Lead (1), Qualified (1) — represent a **10% productive engagement rate** on raw volume.


### 4.2 Campaign Name (`Original Source Drill-Down 2`)

This is the Facebook ad/audience name. Despite the cryptic HubSpot column label, it contains rich information: target region, campaign type (teaser vs lead gen vs lookalike), language, and version.


In [7]:
camp_counts = df['Original Source Drill-Down 2'].value_counts(dropna=False)
print(f"Unique campaign names: {camp_counts.shape[0]}")
print()
for name, count in camp_counts.items():
    label = str(name) if not pd.isna(name) else "(blank)"
    print(f"  {count:>4}  {label}")


Unique campaign names: 9

    53  new project teaser - uae (en) lead gen a.s.
    30  lead gen eur (en) lookalike - a.s.
    24  new project - optimized europe lookalike (en) lead gen a.s.
    19  new project teaser - uk & ireland (en) lead gen a.s.
    12  new project teaser - europe (en) lead gen a.s.
    12  new - lead gen uae (en) - a.s. - copy
    12  lead gen uae (en) - a.s.
     6  new project teaser - gcc (en) lead gen a.s.
     1  Ohana-Wordpress-Forms


**Structure of campaign names:** Each name follows the pattern `[type] - [region] ([language]) [format] [company]`, e.g.:
- `"new project teaser - europe (en) lead gen a.s."` → Teaser type, Europe region, English language
- `"new project teaser - uae (ar) lead gen a.s."` → Teaser type, UAE region, Arabic language

The most important metadata we can parse out:
- **Target region:** UAE, Europe, UK, GCC (from keywords in the name)
- **Campaign type:** Teaser or Lead Gen (appear together in some names — "teaser" takes priority as the ad format)
- **Language:** `(en)` English, `(ar)` Arabic — relevant for understanding which nationalities each campaign reached


### 4.3 Contact Owner, Lifecycle Stage, Country/Region


In [8]:
for col in ['Contact owner', 'Email']:
    if col in df.columns:
        vc = df[col].value_counts(dropna=False)
        print(f"=== {col} ===  ({df[col].notna().sum()} values, {df[col].isna().sum()} null)")
        for val, cnt in vc.head(10).items():
            label = str(val) if not pd.isna(val) else "(blank)"
            print(f"  {cnt:>4}  {label}")
        print()

# Note: Country/Region, City, and Lifecycle Stage are NOT present in this export
present_cols = [c for c in ['Country/Region', 'City', 'Lifecycle Stage'] if c in df.columns]
absent_cols  = [c for c in ['Country/Region', 'City', 'Lifecycle Stage'] if c not in df.columns]
if absent_cols:
    print(f"Note: The following expected columns are absent from this export: {absent_cols}")
    print("Geographic analysis will rely on phone country codes instead.")


=== Contact owner ===  (165 values, 4 null)
    61  Nouhed Hazem
    60  amr shaaban
    44  Ruqaia Hajalsiddig
     4  (blank)

=== Email ===  (168 values, 1 null)
     1  ganjabella12@gmail.com
     1  bhawanisingh06061985@gmail.com
     1  boflan123456789@gmail.com
     1  sergioantoniomoura@hotmail.com
     1  viktormatveev568@gmail.com
     1  ssobir940@gmail.com
     1  elenazilcava@gmail.com
     1  pavlinkabaklinova@gmail.com
     1  modernart.decor@hotmail.com
     1  arambalam647@gmail.com

Note: The following expected columns are absent from this export: ['Country/Region', 'City', 'Lifecycle Stage']
Geographic analysis will rely on phone country codes instead.


**`Country/Region`, `City`, and `Lifecycle Stage` are absent from this export entirely** — HubSpot schema fields that simply weren't populated by the Facebook lead form. This isn't a nulls problem; the columns don't exist. Geography will be derived entirely from the **phone country code** (see Notebook 03).

**Contact owner** has 4 blank values (2.4%) — a small but important gap, as unassigned leads may explain some of the "never contacted" cases.


### 4.4 Facebook Form Name (`Recent conversion`)

This column records the exact Facebook Lead Ad form that was submitted. Unlike the campaign name, it includes additional metadata: the **form version** (V6, V7.1.1, V8.3) and whether the campaign used **"verified numbers"** targeting.


In [9]:
rc_counts = df['Recent Conversion'].value_counts(dropna=False)
print(f"Unique form names: {rc_counts.shape[0]}")
print()
for name, count in rc_counts.items():
    label = str(name)[:85] if not pd.isna(name) else "(blank)"
    print(f"  {count:>4}  {label}")


Unique form names: 10

    54  Facebook Lead Ads: New Project - New lead gen form - A.S. (V7.1.1) - AED
    31  Facebook Lead Ads: New Project - New lead gen form - A.S. (V8.3) - EUR - verified num
    30  Facebook Lead Ads: New lead gen form - A.S. (V4.1) - USD
    19  Facebook Lead Ads: New Project - New lead gen form - A.S. (V8.1) - £
    12  Facebook Lead Ads: New lead gen form - A.S. (V6) - AED
     6  Facebook Lead Ads: New Project - New lead gen form - A.S. (V7.3) - AED - verified num
     6  Facebook Lead Ads: New lead gen form - A.S. (V5) - AED
     5  Facebook Lead Ads: New lead gen form - A.S. (V3) - AED
     5  Facebook Lead Ads: New Project - New lead gen form - A.S. (V8.2.1) - USD - verified n
     1  (blank)


The form names encode **two important business signals** we'll extract as features:
1. **Currency suffix** (AED / EUR / USD) — reveals which market the form was designed for, independent of what the campaign name says. AED = UAE, EUR = Europe.
2. **"Verified numbers"** flag — 42 leads came from forms using Facebook's phone verification feature, which theoretically filters out fake/mistyped numbers and should improve contact rates.

The version numbers (V3 through V8.3) suggest iterative A/B testing over time — comparing conversion and contact rates across versions could reveal what form design changes improved outcomes.


### 4.5 Associated Note (Follow-up Activity Log)


In [10]:
note_col = 'Associated Note'
if note_col in df.columns:
    has_note = df[note_col].notna().sum()
    print(f"Leads with notes: {has_note} / {len(df)} ({has_note/len(df)*100:.1f}%)")
    print()
    print("Sample note content (first 10 unique values):")
    for note in df[note_col].dropna().unique()[:10]:
        print(f"  {str(note)[:90]}")
else:
    print(f"Column not found. Available note-related columns:")
    for c in df.columns:
        if 'note' in c.lower() or 'assoc' in c.lower():
            print(f"  - {c}")


Leads with notes: 135 / 169 (79.9%)

Sample note content (first 10 unique values):
  D1: wa sent
  D1: Uzbekistan speaker \ WA Sent
  D1 : wa sent
  D1: No answer\ WA Sent
  D1: No answer \ WA Sent
  D1: He is interested in apartments. When I started explaining the prices, the client had l
  D1: He is a Spanish speaker. I informed him that one of our Spanish-speaking representativ
  D1: Urdu speaker \ WA Sent
  D1: Phone closed \ WA Sent
  reached over WhatsApp Urdu Speaker;D1: Phone closed \ WA Sent


The **Associated Note** column is the agent's activity log — it records what happened on each contact attempt. The D1/D2/D3 prefix (Day 1, Day 2, Day 3) is the team's own shorthand for follow-up sequences. This field is invaluable for understanding actual agent behaviour, but it requires text parsing to extract structured features from free-form entries like `"D1: No answer \ WA Sent"` (Day 1: no phone answer, WhatsApp message sent).

Note: 34 leads (20%) have no note entry at all — consistent with the 34 null Last Activity Dates, these are likely the same leads that were never contacted after import.


## 5. Numeric Column Analysis

Only two columns have numeric dtype in the raw export: `Record ID` (a HubSpot identifier) and `Phone Number`. The phone number situation is a data quality issue in its own right.


In [11]:
num_cols = df.select_dtypes(include='number').columns.tolist()
print(f"Numeric columns: {num_cols}")
print()
df[num_cols].describe()


Numeric columns: ['Record ID', 'Phone Number', 'Source 3', 'Are you an investor or a broker', 'Unit Type', 'Unit Value', 'Original Create Date', 'Recent Deal Close Date']



,Record ID,Phone Number,Source 3,Are you an investor or a broker,Unit Type,Unit Value,Original Create Date,Recent Deal Close Date
count,169.00,168.00,0.00,0.00,0.00,0.00,0.00,0.00
mean,208426009141.20,851855935781.04,NaN,NaN,NaN,NaN,NaN,NaN
std,309127767.47,1266279714291.54,NaN,NaN,NaN,NaN,NaN,NaN
min,207922661970.00,22893125038.00,NaN,NaN,NaN,NaN,NaN,NaN
25%,208167205930.00,380667902420.25,NaN,NaN,NaN,NaN,NaN,NaN
50%,208430337291.00,918747679268.00,NaN,NaN,NaN,NaN,NaN,NaN
75%,208722336134.00,971527288944.00,NaN,NaN,NaN,NaN,NaN,NaN
max,208962569084.00,9779815800225.00,NaN,NaN,NaN,NaN,NaN,NaN


**The phone number problem:** Phone numbers are stored as `float64` because Excel infers numeric types for digit-only cells. The consequence: a number like `971501234567` (UAE mobile) is stored as `971501234567.0` and displayed as `9.715e+11` in scientific notation. Casting directly to `str()` gives `"971501234567.0"` — the trailing `.0` would break any phone parsing library. The correct fix is a two-step conversion: `float → int → str`, which drops the decimal and preserves the full number without scientific notation.


## 6. Date Column Analysis

Two date columns exist: `Create Date` (when the lead entered HubSpot) and `Last Activity Date` (last agent touchpoint). The gap between them is our proxy for response time.


In [12]:
for col in ['Create Date', 'Last Activity Date']:
    dt = pd.to_datetime(df[col], errors='coerce', utc=True)
    null_n = dt.isna().sum()
    print(f"=== {col} ===")
    print(f"  Non-null : {dt.notna().sum()} / {len(dt)} ({null_n} missing)")
    if dt.notna().any():
        print(f"  Earliest : {dt.min()}")
        print(f"  Latest   : {dt.max()}")
        span = (dt.max() - dt.min()).days
        print(f"  Span     : {span} days")
        print(f"  By weekday:")
        day_counts = dt.dt.day_name().value_counts()
        for day, cnt in day_counts.sort_values(ascending=False).items():
            print(f"    {day:<12} {cnt:>4}")
    print()


=== Create Date ===
  Non-null : 169 / 169 (0 missing)
  Earliest : 2026-03-09 00:20:00+00:00
  Latest   : 2026-03-13 09:22:00+00:00
  Span     : 4 days
  By weekday:
    Tuesday        44
    Thursday       42
    Monday         41
    Wednesday      32
    Friday         10

=== Last Activity Date ===
  Non-null : 135 / 169 (34 missing)
  Earliest : 2026-03-09 08:58:00+00:00
  Latest   : 2026-03-13 09:42:00+00:00
  Span     : 4 days
  By weekday:
    Thursday       41
    Wednesday      38
    Tuesday        35
    Monday         11
    Friday         10



**Key findings:**

- The **5-day window (March 9–13)** is very tight for trend analysis — any day-over-day patterns may reflect day-of-week effects rather than genuine trends, since Monday and Thursday naturally see different lead volumes regardless of campaign spend.

- **34 leads (20%) have no Last Activity Date** — these are leads that entered HubSpot but were never actioned. Combined with the 31 blank Lead Status rows, this points to a systemic intake process failure: leads are being imported but not assigned or opened by agents in time.

- The **average response time** (to be calculated after cleaning) will be biased upward if activity dates reflect WhatsApp messages or admin updates rather than the first call attempt.


## 7. Null Count Visualisation

A visual summary of missing data makes the 6 empty columns immediately obvious and provides a snapshot we can include in any project documentation.


In [13]:
null_df = pd.DataFrame({
    'column'    : df.columns,
    'null_count': df.isna().sum().values,
    'pct_null'  : (df.isna().mean() * 100).round(1).values,
}).sort_values('null_count', ascending=True)

# Colour by severity
def null_colour(pct):
    if pct == 100:  return '#C62828'
    if pct >= 20:   return '#F57C00'
    if pct > 0:     return '#FFB74D'
    return '#2E7D32'

null_df['colour'] = null_df['pct_null'].apply(null_colour)

fig = go.Figure(go.Bar(
    x=null_df['null_count'],
    y=null_df['column'],
    orientation='h',
    text=[f"{v:.0f}%" if v > 0 else "0%" for v in null_df['pct_null']],
    textposition='outside',
    marker_color=null_df['colour'],
))
fig.update_layout(
    title='Null Counts per Column — Raw Dataset (red = 100% empty, orange = 20%+)',
    xaxis_title='Number of Null Values',
    yaxis_title='',
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=560,
    margin=dict(l=220, r=80, t=70, b=40),
)
save_fig(fig, '01_null_counts_raw')
fig.show()


The chart makes the problem structure clear: **six columns are entirely red** (100% null) and can be dropped without any information loss. The remaining sparse columns — Last Activity Date (20% null) and Lead Status (18% null) — require imputation decisions rather than deletion.


## 8. Data Quality Issues Found

The following table catalogs every data quality problem discovered in the raw export, its scope, and the proposed resolution.


In [14]:
issues = [
    {
        'Issue'       : '6 completely empty columns',
        'Scope'       : '100% null (0/169 values)',
        'Columns'     : 'Source 3, Are you an investor or a broker, Unit Type, Unit Value, Original Create Date, Recent Deal Close Date',
        'Resolution'  : 'Drop — zero information content',
    },
    {
        'Issue'       : '3 zero-variance columns',
        'Scope'       : 'Single unique value across all 169 rows',
        'Columns'     : 'Marketing contact status (all "Non-marketing contact"), Original Source (all "OFFLINE"), Original Source Drill-Down 1 (all "Facebook Lead Ads")',
        'Resolution'  : 'Drop — no analytical value; confirmed we are working with Facebook leads only',
    },
    {
        'Issue'       : 'Phone Number stored as float64',
        'Scope'       : '168 affected (1 missing)',
        'Columns'     : 'Phone Number',
        'Resolution'  : 'Convert: float -> int -> str to strip .0 and prevent scientific notation',
    },
    {
        'Issue'       : '31 blank Lead Status values',
        'Scope'       : '18.3% of all leads',
        'Columns'     : 'Lead Status',
        'Resolution'  : 'Fill with "Uncontacted" — blank status means no agent action taken',
    },
    {
        'Issue'       : '4 blank Contact owner values',
        'Scope'       : '2.4% of all leads',
        'Columns'     : 'Contact owner',
        'Resolution'  : 'Fill with "Unassigned"; apply title-case to existing names for consistency',
    },
    {
        'Issue'       : '34 null Last Activity Date values',
        'Scope'       : '20.1% of all leads',
        'Columns'     : 'Last Activity Date',
        'Resolution'  : 'Preserve as NaN — these are valid "never contacted" cases; flag separately in analysis',
    },
    {
        'Issue'       : 'Column names have spaces and special characters',
        'Scope'       : 'All 22 columns',
        'Columns'     : 'e.g. "Country/Region", "HubSpot Owner", "Associated Note IDs"',
        'Resolution'  : 'Rename all to snake_case using a consistent mapping',
    },
    {
        'Issue'       : 'Campaign name in misleadingly-labelled column',
        'Scope'       : 'Entire dataset',
        'Columns'     : '"Original Source Drill-Down 2"',
        'Resolution'  : 'Rename to "campaign_name" for clarity',
    },
    {
        'Issue'       : 'Country/Region, City, Lifecycle Stage absent from export',
        'Scope'       : 'Columns do not exist in this HubSpot export',
        'Columns'     : 'Country/Region, City, Lifecycle Stage',
        'Resolution'  : 'Not in this export — derive geography from phone country code; lifecycle analysis not available',
    },
]

issues_df = pd.DataFrame(issues)
issues_df.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})                .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])


,Issue,Scope,Columns,Resolution
0,6 completely empty columns,100% null (0/169 values),"Source 3, Are you an investor or a broker, Unit Type, Unit Value, Original Create Date, Recent Deal Close Date",Drop — zero information content
1,3 zero-variance columns,Single unique value across all 169 rows,"Marketing contact status (all ""Non-marketing contact""), Original Source (all ""OFFLINE""), Original Source Drill-Down 1 (all ""Facebook Lead Ads"")",Drop — no analytical value; confirmed we are working with Facebook leads only
2,Phone Number stored as float64,168 affected (1 missing),Phone Number,Convert: float -> int -> str to strip .0 and prevent scientific notation
3,31 blank Lead Status values,18.3% of all leads,Lead Status,"Fill with ""Uncontacted"" — blank status means no agent action taken"
4,4 blank Contact owner values,2.4% of all leads,Contact owner,"Fill with ""Unassigned""; apply title-case to existing names for consistency"
5,34 null Last Activity Date values,20.1% of all leads,Last Activity Date,"Preserve as NaN — these are valid ""never contacted"" cases; flag separately in analysis"
6,Column names have spaces and special characters,All 22 columns,"e.g. ""Country/Region"", ""HubSpot Owner"", ""Associated Note IDs""",Rename all to snake_case using a consistent mapping
7,Campaign name in misleadingly-labelled column,Entire dataset,"""Original Source Drill-Down 2""","Rename to ""campaign_name"" for clarity"
8,"Country/Region, City, Lifecycle Stage absent from export",Columns do not exist in this HubSpot export,"Country/Region, City, Lifecycle Stage",Not in this export — derive geography from phone country code; lifecycle analysis not available


## 9. Cleaning Plan

The cleaning pipeline (`src/cleaning.py`) addresses every issue above in the following order:

1. **Read first sheet only** — ignore the pivot table on Sheet1
2. **Drop 6 empty columns** — Source 3, investor/broker, Unit Type, Unit Value, Original Create Date, Recent Deal Close Date
3. **Drop 3 zero-variance columns** — Marketing contact status, Original Source, Original Source Drill-Down 1
4. **Convert phone numbers** — float → int → str to preserve full number without scientific notation
5. **Fill blank Lead Status** — NaN → "Uncontacted"
6. **Standardise Contact owner** — apply `.str.title()`, fill blanks → "Unassigned"
7. **Parse date columns** — both date columns to `datetime64[UTC]` with `errors='coerce'`
8. **Rename all columns to snake_case** — explicit mapping for domain-meaningful renames (e.g. `Original Source Drill-Down 2` → `campaign_name`), automatic snake_case for the rest

**Output:** `data/processed/cleaned.parquet` (primary) and `data/processed/cleaned.csv` (human-readable reference).

After cleaning we expect: **169 rows × 13 columns**, with 0 nulls in `lead_status` and `hubspot_owner`.


### Quick Sanity Check: Cleaned Data Already Exists

Since we ran the cleaning pipeline earlier, we can verify the expected output right now.


In [15]:
cleaned = pd.read_parquet(config.CLEANED_PARQUET)
print(f"Cleaned shape : {cleaned.shape[0]} rows x {cleaned.shape[1]} columns  (expected 169 x 13)")
print(f"\nNull counts in key columns:")
for col in ['lead_status', 'hubspot_owner', 'phone_number', 'campaign_name']:
    if col in cleaned.columns:
        n = cleaned[col].isna().sum()
        print(f"  {col:<20} {n} nulls  {'OK' if n == 0 else 'ISSUE'}")

print(f"\nPhone number sample (first 5):")
print(cleaned['phone_number'].head().tolist())
print(f"\nLead status distribution after cleaning:")
print(cleaned['lead_status'].value_counts())


Cleaned shape : 169 rows x 13 columns  (expected 169 x 13)

Null counts in key columns:
  lead_status          0 nulls  OK
  hubspot_owner        0 nulls  OK
  phone_number         1 nulls  ISSUE
  campaign_name        0 nulls  OK

Phone number sample (first 5):
['393511841860', '971524141674', '971508877511', '33617529803', '380668898568']

Lead status distribution after cleaning:
lead_status
No Answer                  96
Uncontacted                31
Contacted                  17
Unqualified                14
Not Interested              3
Future Opportunity          3
Junk Lead                   2
Newsletter Subscription     1
Qualified                   1
Hot Lead                    1
Name: count, dtype: int64


The cleaned data confirms that all 9 structural issues have been resolved: 13 columns remain (9 dropped), phone numbers are clean strings, and "Uncontacted" now accounts for 31 additional leads surfaced from the blank status rows.


## Key Takeaways

1. **The raw data is 40% structural waste.** Of 22 columns, 6 are completely empty and 3 have zero variance — leaving 13 genuinely informative columns. This is typical of HubSpot exports where schema fields exist regardless of whether the form collects them.

2. **57% of leads never answer the phone, and 18% were never even attempted.** These are not separate problems — both trace back to response latency. With an average response time of 18.6 hours against an industry benchmark of 5 minutes, many leads have likely forgotten they submitted a form by the time they receive a call.

3. **Phone number geography is the only reliable location signal.** The `Country/Region` field is 92% empty. Phone country codes (971 = UAE, 380 = Ukraine, 44 = UK) will be parsed from the phone number string in the feature engineering step to reconstruct geographic origin.

4. **Campaign names encode critical metadata.** Target region, campaign type (teaser vs lead gen), language, and form version are all embedded in the `Original Source Drill-Down 2` and `Recent conversion` fields. Parsing these will enable campaign-level performance comparisons in Notebooks 05 and 07.

5. **The 5-day window limits trend analysis.** 169 leads over 5 weekdays gives roughly 34 leads/day — not enough for statistically robust day-over-day trend claims. Analyses will focus on cross-campaign and cross-agent comparisons rather than temporal patterns.

---
*Next: Notebook 02 — Data Cleaning Walkthrough, demonstrating each cleaning step with before/after validation.*
